In [ ]:
score_flags = su.execute_custom_query_gp(score_flags_query)

scores = pd.concat(
    [
        march_scores.assign(score_month='Март', mrc_desc='ZTAN57'),
        april_scores.assign(score_month='Апрель', mrc_desc='ZTANTTT63')
    ],
    ignore_index=True
)

scores['score'] = pd.to_numeric(scores['score'], errors='coerce')

scores = scores.drop_duplicates(
    subset=['score_month', 'client_id']
)

merged_scores = scores.merge(
    score_flags[
        [
            'contact_id',
            'mrc_desc',
            'subscribed_period_flg',
            'view_offer_flg',
            'subscribed_after_view_flg'
        ]
    ],
    left_on=['client_id', 'mrc_desc'],
    right_on=['contact_id', 'mrc_desc'],
    how='left'
)

flag_cols = [
    'subscribed_period_flg',
    'view_offer_flg',
    'subscribed_after_view_flg'
]

merged_scores[flag_cols] = (
    merged_scores[flag_cols]
    .fillna(0)
    .astype(int)
)

merged_scores['score_bin'] = pd.cut(
    merged_scores['score'],
    bins=10
)

score_stats = (
    merged_scores
    .groupby(['score_month', 'score_bin'], as_index=False, observed=True)
    .agg(
        avg_score=('score', 'mean'),
        clients_cnt=('client_id', 'nunique'),
        subscribed_cnt=('subscribed_period_flg', 'sum'),
        view_offer_cnt=('view_offer_flg', 'sum'),
        subscribed_after_view_cnt=('subscribed_after_view_flg', 'sum')
    )
)

score_stats['subscribed_pct'] = (
    score_stats['subscribed_cnt'] / score_stats['clients_cnt'] * 100
).round(2)

score_stats['view_offer_pct'] = (
    score_stats['view_offer_cnt'] / score_stats['clients_cnt'] * 100
).round(2)

score_stats['subscribed_after_view_pct'] = (
    score_stats['subscribed_after_view_cnt'] / score_stats['clients_cnt'] * 100
).round(2)

score_stats['score_bin'] = score_stats['score_bin'].apply(
    lambda x: f'{x.left:.2f} - {x.right:.2f}'
)

score_stats